# Test SQL Queries

In [1]:
# Database
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri(r"sqlite:///C:\Users\peter\Documents\SJSU\Thesis\code\mini_dev\minidev_0703\minidev\MINIDEV\dev_databases\debit_card_specializing\debit_card_specializing.sqlite")

In [16]:
from langchain_core.tools import tool
import ast
from textwrap import dedent

# Define tools
@tool
def run_statement() -> str:
    """Retrieves all available tables in database by running "SHOW TABLES;" statement.
    
    Args:
        No Args
    """
    query = f"SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"
    tables = db.run(query)
    
    tables_list = ast.literal_eval(tables)

    database_info = []

    for table_tuple in tables_list:
        table_name = table_tuple[0]
        
        # Get schema
        schema_query = f"PRAGMA table_info({table_name});"
        schema = db.run(schema_query)
        
        # Get first 3 rows
        sample_query = f"SELECT * FROM {table_name} LIMIT 3;"
        sample_data = db.run(sample_query)
        
        # Format for LLM
        table_info = dedent(f"""
            Table: {table_name}
            Schema: {schema}
            Sample Data (first 3 rows): {sample_data}
            ---
            """)
        database_info.append(table_info)

    # Combine all table information
    formatted_output = "\n".join(database_info)

    return formatted_output

run_statement.invoke({})

"\nTable: customers\nSchema: [(0, 'CustomerID', 'INTEGER', 1, None, 1), (1, 'Segment', 'TEXT', 0, None, 0), (2, 'Currency', 'TEXT', 0, None, 0)]\nSample Data (first 3 rows): [(3, 'SME', 'EUR'), (5, 'LAM', 'EUR'), (6, 'SME', 'EUR')]\n---\n\n\nTable: gasstations\nSchema: [(0, 'GasStationID', 'INTEGER', 1, None, 1), (1, 'ChainID', 'INTEGER', 0, None, 0), (2, 'Country', 'TEXT', 0, None, 0), (3, 'Segment', 'TEXT', 0, None, 0)]\nSample Data (first 3 rows): [(44, 13, 'CZE', 'Value for money'), (45, 6, 'CZE', 'Premium'), (46, 23, 'CZE', 'Other')]\n---\n\n\nTable: products\nSchema: [(0, 'ProductID', 'INTEGER', 1, None, 1), (1, 'Description', 'TEXT', 0, None, 0)]\nSample Data (first 3 rows): [(1, 'Rucní zadání'), (2, 'Nafta'), (3, 'Special')]\n---\n\n\nTable: transactions_1k\nSchema: [(0, 'TransactionID', 'INTEGER', 0, None, 1), (1, 'Date', 'DATE', 0, None, 0), (2, 'Time', 'TEXT', 0, None, 0), (3, 'CustomerID', 'INTEGER', 0, None, 0), (4, 'CardID', 'INTEGER', 0, None, 0), (5, 'GasStationID', 'IN